## Postprocessing

See documentation [here](https://www.notion.so/Full-Logs-Documentation-f4a90b2f0514493e9fad210e72f19b23?source=copy_link#28dc697d927c8034a709dce2b397b27b).

In [1]:
import json
import pandas as pd

# --- Load original test data ---
with open("../data/SROIE2019/test/test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

# --- Load predictions ---
with open("../data/SROIE2019/test/predictions.json", "r", encoding="utf-8") as f:
    predictions_data = json.load(f)

# --- Load ground-truth invoice numbers ---
with open("../data/SROIE2019/test/test_labels.json", "r", encoding="utf-8") as f:
    test_labels_data = json.load(f)

# --- Create DataFrame rows ---
rows = []

# Create a lookup for test data by file
test_lookup = {item["file"]: item for item in test_data}

for pred_item in predictions_data:
    file_name = pred_item["file"]
    labels = pred_item["predicted_labels"]
    
    # Get corresponding words and bboxes
    if file_name not in test_lookup:
        continue  # safety check
    words = test_lookup[file_name]["words"]
    bboxes = test_lookup[file_name]["bboxes"]
    
    # Make sure lengths match
    if len(words) != len(labels) or len(words) != len(bboxes):
        print(f"Warning: Length mismatch in {file_name}")
        min_len = min(len(words), len(labels), len(bboxes))
        words, labels, bboxes = words[:min_len], labels[:min_len], bboxes[:min_len]
    
    # Append each token as a row
    for word, bbox, label in zip(words, bboxes, labels):
        rows.append({
            "file": file_name,
            "word": word,
            "bbox": bbox,
            "prediction": label
        })

# --- Create DataFrame ---
df = pd.DataFrame(rows)
df["true_invoice_number"] = df["file"].map(test_labels_data)

# filter to where prediction is not Label_0
df_filtered = df[df["prediction"] != "LABEL_0"]
df_filtered

,file,word,bbox,prediction,true_invoice_number
39,X51005675104.jpg,CS00012944,"[319, 369, 547, 388]",LABEL_1,CS00012944
205,X51005568890.jpg,66549,"[170, 390, 284, 413]",LABEL_1,66549
375,X00016469670.jpg,PEGIV,"[291, 360, 393, 383]",LABEL_2,PEGIV-1030765
376,X00016469670.jpg,-,"[393, 360, 414, 383]",LABEL_2,PEGIV-1030765
377,X00016469670.jpg,1030765,"[414, 360, 559, 383]",LABEL_2,PEGIV-1030765
...,...,...,...,...,...
62870,X51005763958.jpg,152164,"[343, 454, 459, 473]",LABEL_1,152164
63025,X51006912959.jpg,STU001,"[409, 232, 452, 244]",LABEL_1,STU001/59572
63026,X51006912959.jpg,/,"[452, 232, 459, 244]",LABEL_2,STU001/59572
63027,X51006912959.jpg,59572,"[459, 232, 495, 244]",LABEL_2,STU001/59572


In [2]:
def combine_invoice_numbers(df):
    """
    Function to combine invoice tokens for each file
    """
    combined_rows = []

    for file_name, group in df.groupby("file"):
        invoice_tokens = []
        current_invoice = []
        last_label = None

        for _, row in group.iterrows():
            label = row["prediction"]
            word = row["word"]

            # Only consider invoice labels
            if label in ["LABEL_1", "LABEL_2"]:
                # Start a new invoice if LABEL_1
                if label == "LABEL_1":
                    if current_invoice:
                        invoice_tokens.append(" ".join(current_invoice))
                    current_invoice = [word]
                else:  # LABEL_2
                    current_invoice.append(word)
            else:
                # If sequence ends
                if current_invoice:
                    invoice_tokens.append(" ".join(current_invoice))
                    current_invoice = []

        # Append any remaining invoice tokens
        if current_invoice:
            invoice_tokens.append(" ".join(current_invoice))

        # Store one row per file with combined invoice number prediction
        combined_rows.append({
            "file": file_name,
            "predicted_invoice_number": " ".join(invoice_tokens),
            "true_invoice_number": group["true_invoice_number"].iloc[0]  # same for all rows
        })

    combined_df = pd.DataFrame(combined_rows)
    return combined_df

invoice_df = combine_invoice_numbers(df)
invoice_df["correct"] = invoice_df["predicted_invoice_number"] == invoice_df["true_invoice_number"]
invoice_df

,file,predicted_invoice_number,true_invoice_number,correct
0,X00016469670.jpg,PEGIV - 1030765,PEGIV-1030765,False
1,X00016469671.jpg,PEGIV - 1030531,PEGIV-1030531,False
2,X51005200931.jpg,CS00082258,CS00082258,True
3,X51005230605.jpg,19729058 129077,19729058,False
4,X51005230616.jpg,141900016842,141900016842,True
...,...,...,...,...
342,X51008099100.jpg,276703,276703,True
343,X51009008095.jpg,CS180319 - 0015,CS180319-0015,False
344,X51009447842.jpg,CR0008955,CR0008955,True
345,X51009453729.jpg,: CS - 20242,CS-20242,False


In [3]:
# --- Overall accuracy ---
accuracy = invoice_df["correct"].mean()
print(f"Exact match accuracy: {accuracy:.2%}")

# --- Show mismatches ---
mismatches = invoice_df[invoice_df["correct"] == False]
print(f"Number of mismatches: {len(mismatches)}")
mismatches[["file", "predicted_invoice_number", "true_invoice_number"]]

# mismatches without ambiguous
mismatches_no_ambiguous = invoice_df[~invoice_df["true_invoice_number"].str.contains("ambiguous")]
print(f"Accuracy w/o ambiguous labels: {mismatches_no_ambiguous['correct'].mean():.2%}")


Exact match accuracy: 58.79%
Number of mismatches: 143
Accuracy w/o ambiguous labels: 63.16%


## Heuristics for postprocessing predictions

In [4]:
print(invoice_df[invoice_df["predicted_invoice_number"].str.contains(":")])

                 file predicted_invoice_number true_invoice_number  correct
108  X51005745213.jpg             : CS - 51762            CS-51762    False
345  X51009453729.jpg             : CS - 20242            CS-20242    False


In [5]:
# remove : and strip from predicted invoice numbers
invoice_df["predicted_invoice_number"] = invoice_df["predicted_invoice_number"].str.replace(":", "").str.strip()

# if contains ' - ', replace by empty string
invoice_df["predicted_invoice_number"] = invoice_df["predicted_invoice_number"].str.replace(" - ", "-")

# if contains ' / ' replace by empty string
invoice_df["predicted_invoice_number"] = invoice_df["predicted_invoice_number"].str.replace(" / ", "/")

# replace SP NULL with SP-NULL
invoice_df["predicted_invoice_number"] = invoice_df["predicted_invoice_number"].str.replace("SP NULL", "SP-NULL")

# if contains SP-NULL, only take the ifrst 16 characters
invoice_df["predicted_invoice_number"] = invoice_df["predicted_invoice_number"].apply(lambda x: x[:24] if "SP-NULL" in x else x)

# if contains the string DATE at the end, remove it
invoice_df["predicted_invoice_number"] = invoice_df["predicted_invoice_number"].apply(lambda x: x[:-4].strip() if x.endswith("DATE") else x)

# --- Overall accuracy ---
invoice_df["correct"] = invoice_df["predicted_invoice_number"] == invoice_df["true_invoice_number"]
accuracy = invoice_df["correct"].mean()
print(f"Exact match accuracy: {accuracy:.2%}")

# --- Show mismatches ---
mismatches = invoice_df[invoice_df["correct"] == False]
print(f"Number of mismatches: {len(mismatches)}")
mismatches[["file", "predicted_invoice_number", "true_invoice_number"]]

Exact match accuracy: 75.79%
Number of mismatches: 84


,file,predicted_invoice_number,true_invoice_number
3,X51005230605.jpg,19729058 129077,19729058
9,X51005268275.jpg,,LCN00212
11,X51005288570.jpg,3180301 K0131800235697,K0131800235697
12,X51005301666.jpg,,LCN00212
16,X51005361908.jpg,CS1803/28617 4974052801334,CS1803/28617
...,...,...,...
285,X51007231346.jpg,,ambiguous
306,X51007579725.jpg,001-1112563-1112563,001-1112563
324,X51007846379.jpg,403149******8937 2018060310100050214,2018060310100050214
325,X51007846387.jpg,MR-T01105105-T01105105 9555047308127,MR-T01105105


In [6]:
# calculate accuracy excluding where true_invoice_number is 'ambiguous'
filtered_df = invoice_df[invoice_df["true_invoice_number"] != "ambiguous"]
accuracy = filtered_df["correct"].mean()
print(f"Exact match accuracy (excluding 'ambiguous'): {accuracy:.2%}")

Exact match accuracy (excluding 'ambiguous'): 81.42%


In [7]:
# find mismatches where true_invoice_number is not 'ambiguous'
mismatches = invoice_df[(invoice_df["correct"] == False) & (invoice_df["true_invoice_number"] != "ambiguous") & (invoice_df["predicted_invoice_number"] != "") ]
print(f"Number of mismatches (excluding 'ambiguous'): {len(mismatches)}")
mismatches[["file", "predicted_invoice_number", "true_invoice_number"]]

Number of mismatches (excluding 'ambiguous'): 48


,file,predicted_invoice_number,true_invoice_number
3,X51005230605.jpg,19729058 129077,19729058
11,X51005288570.jpg,3180301 K0131800235697,K0131800235697
16,X51005361908.jpg,CS1803/28617 4974052801334,CS1803/28617
18,X51005361923.jpg,002017808384 0100036262,0100036262
22,X51005433548.jpg,001-1541798-1541798,001-1541798
23,X51005433556.jpg,1000249-1220845,003-1220845
27,X51005442366.jpg,- 0014888-0014888,REC-0014888
33,X51005444044.jpg,18332/103/T0157 570285,18332/103/T0157
40,X51005568855.jpg,9050338975 2002008362249,9050338975
42,X51005568885.jpg,139110 01 -,139110


In [8]:

# filter to predicted_invoice_number if contains spaces or dash
# filter mismatches if contains two dashes in string

## IF ONLY CONTAIN LETTERS, NUMBERS, DASHES, SLASHES, +, OR SPACES --> REPLACE BY EMPTY STRING
# Contains spaces --> human review
## NEXT STEP, playing around with entities and rules to improve accuracy

# begin or end with special characters --> human review

# after sending humans to review, what is left


filtered_mismatches = mismatches[~mismatches["predicted_invoice_number"].str.contains(" ") &
 ~mismatches["predicted_invoice_number"].str.contains("/") &
  ~(mismatches["predicted_invoice_number"].str.count("-") > 1)]
print(filtered_mismatches)
print(f'{len(filtered_mismatches)} / {len(invoice_df)} ({len(filtered_mismatches) / len(invoice_df):.2%}) wrong after sending to review')

                 file predicted_invoice_number true_invoice_number  correct
23   X51005433556.jpg          1000249-1220845         003-1220845    False
86   X51005715007.jpg      0000MTW-P2000086706          1502102697    False
89   X51005719855.jpg                  A063975              063975    False
187  X51006414715.jpg                   286815            P1176502    False
4 / 347 (1.15%) wrong after sending to review


## Ablation study on heuristics

In [9]:
invoice_df[(invoice_df['correct'] == True) & ~(invoice_df['predicted_invoice_number'].str.contains(" "))]

# how many single lines passed

# accuracy of those that do not contain spaces and are correct
len(invoice_df[(invoice_df['correct'] == True) & ~(invoice_df['predicted_invoice_number'].str.contains(" "))]) / len(invoice_df[~(invoice_df['predicted_invoice_number'].str.contains(" "))])


0.8486842105263158

In [10]:
invoice_df[(invoice_df['correct'] == True) & (invoice_df['predicted_invoice_number'].str.contains(" "))]

,file,predicted_invoice_number,true_invoice_number,correct
56,X51005663274.jpg,CS 24358,CS 24358,True
57,X51005663300.jpg,CS 24146,CS 24146,True
60,X51005663310.jpg,CS 24388,CS 24388,True
160,X51006349081.jpg,CR 1804/1627,CR 1804/1627,True
163,X51006350737.jpg,CR 1803/1905,CR 1803/1905,True


In [11]:
mismatches[mismatches["predicted_invoice_number"].str.contains("CS")]
# if CS + space --> OK

# Filter out bad examples first

,file,predicted_invoice_number,true_invoice_number,correct
16,X51005361908.jpg,CS1803/28617 4974052801334,CS1803/28617,False
135,X51006008082.jpg,CS00534185 CS00530344,CS00534185,False
152,X51006332641.jpg,CS00027489 2411662341,CS00027489,False
273,X51007103597.jpg,CS -,CS-0014464,False


In [12]:
mismatches[mismatches["predicted_invoice_number"].str.contains("CR")]

,file,predicted_invoice_number,true_invoice_number,correct


## Checking ambiguous labels

In [13]:
# filter where true_invoice_number is 'ambiguous'
invoice_df[invoice_df["true_invoice_number"] == "ambiguous"]

,file,predicted_invoice_number,true_invoice_number,correct
30,X51005442388.jpg,,ambiguous,False
51,X51005587261.jpg,,ambiguous,False
75,X510056849111.jpg,,ambiguous,False
76,X51005684949.jpg,,ambiguous,False
95,X51005719898.jpg,,ambiguous,False
119,X51005757292.jpg,,ambiguous,False
120,X51005757308.jpg,,ambiguous,False
121,X51005757342.jpg,,ambiguous,False
191,X51006466070.jpg,,ambiguous,False
229,X51006619346.jpg,,ambiguous,False


## Check empty predictions

In [14]:
print(len(invoice_df[(invoice_df["predicted_invoice_number"] == "") & (invoice_df["true_invoice_number"] != "ambiguous")]))
invoice_df[(invoice_df["predicted_invoice_number"] == "") & (invoice_df["true_invoice_number"] != "ambiguous")] 

12


,file,predicted_invoice_number,true_invoice_number,correct
9,X51005268275.jpg,,LCN00212,False
12,X51005301666.jpg,,LCN00212,False
24,X51005442322.jpg,,65991,False
117,X51005757233.jpg,,68089,False
144,X51006311714.jpg,,53488,False
169,X51006388068.jpg,,00600696340036805,False
174,X51006392167.jpg,,SAB01201706190120,False
255,X51006647933.jpg,,2,False
256,X51006647984.jpg,,21485-00,False
258,X51006828199.jpg,,6186,False
